<a href="https://colab.research.google.com/github/YesidCruz/data-analyst-portfolio/blob/main/Limpiando_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# URL directa a un CSV grande de ejemplo (puedes reemplazarla por otro dataset público)
url = "https://raw.githubusercontent.com/plotly/datasets/master/2016-weather-data-seattle.csv"

# Cargar datos
df = pd.read_csv(url)

# Ver primeras filas
df.head()

# Información general
df.info()

# Estadísticas descriptivas
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24381 entries, 0 to 24380
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date               24381 non-null  object 
 1   Max_TemperatureC   24381 non-null  int64  
 2   Mean_TemperatureC  24376 non-null  float64
 3   Min_TemperatureC   24380 non-null  float64
dtypes: float64(2), int64(1), object(1)
memory usage: 762.0+ KB


,Max_TemperatureC,Mean_TemperatureC,Min_TemperatureC
count,24381.000000,24376.000000,24380.000000
mean,14.944014,11.043485,7.252338
std,7.171926,5.794158,4.913692
min,-18.000000,-12.000000,-17.000000
25%,9.000000,7.000000,4.000000
50%,14.000000,11.000000,7.000000
75%,20.000000,16.000000,11.000000
max,54.000000,31.000000,22.000000


In [ ]:
import pandas as pd
import numpy as np

# 1. Vista rápida
print("Primeras filas:")
display(df.head())

# 2. Estructura y tipos
print("\nInformación del dataset:")
df.info()

# 3. ¿Cuántos valores faltan por columna?
print("\nValores nulos por columna:")
print(df.isnull().sum())

# 4. ¿Filas duplicadas?
print(f"\nFilas duplicadas: {df.duplicated().sum()}")

Primeras filas:


,Date,Max_TemperatureC,Mean_TemperatureC,Min_TemperatureC
0,1/1/1948,10,8.0,7.0
1,1/2/1948,6,4.0,3.0
2,1/3/1948,7,4.0,2.0
3,1/4/1948,7,4.0,2.0
4,1/5/1948,7,3.0,0.0



Información del dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24381 entries, 0 to 24380
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date               24381 non-null  object 
 1   Max_TemperatureC   24381 non-null  int64  
 2   Mean_TemperatureC  24376 non-null  float64
 3   Min_TemperatureC   24380 non-null  float64
dtypes: float64(2), int64(1), object(1)
memory usage: 762.0+ KB

Valores nulos por columna:
Date                 0
Max_TemperatureC     0
Mean_TemperatureC    5
Min_TemperatureC     1
dtype: int64

Filas duplicadas: 0


In [ ]:
df = df.drop_duplicates()
print(f"tamaño tras eliminar duplicados:{df.shape}")

tamaño tras eliminar duplicados:(24381, 4)


In [ ]:
# Opción 1: Borrar filas con muchos nulos (umbral: eliminar si tienen más de 3 nulos)
df = df.dropna(thresh=df.shape[1] - 3)

# Opción 2: Rellenar nulos numéricos con la mediana
cols_num = df.select_dtypes(include=np.number).columns
df[cols_num] = df[cols_num].fillna(df[cols_num].median())

# Opción 3: Rellenar nulos categóricos con la moda (más frecuente)
cols_cat = df.select_dtypes(include='object').columns
for col in cols_cat:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# Si tienes columna de fechas, pásala a datetime
#df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')

# Forzar columnas numéricas (las que tengan comas o signos $)
# df['precio'] = df['precio'].str.replace('$', '').str.replace(',', '').astype(float)

In [ ]:
cols_texto = df.select_dtypes(include='object').columns
for col in cols_texto:
    df[col] = df[col].str.strip()  # Quita espacios al inicio/final
    # Si quieres uniformar en minúsculas (descomenta):
    # df[col] = df[col].str.lower()

In [ ]:
cols_num = df.select_dtypes(include=np.number).columns
for col in cols_num:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    # Los reemplazamos por los límites (capado) en lugar de borrarlos
    #Detectar y tratar outliers (valores extremos) usando el rango intercuartílico (IQR)
    df[col] = df[col].clip(lower=limite_inf, upper=limite_sup)

In [ ]:
# Mira cómo quedaron los nulos ahora
print(df.isnull().sum().sum(), "nulos totales restantes.")
#Verificacion final

# Estadísticas finales
display(df.describe(include='all'))

0 nulos totales restantes.


,Date,Max_TemperatureC,Mean_TemperatureC,Min_TemperatureC
count,24381,24381.000000,24381.00000,24381.000000
unique,24381,NaN,NaN,NaN
top,12/31/2015,NaN,NaN,NaN
freq,1,NaN,NaN,NaN
mean,NaN,14.957836,11.04727,7.267975
std,NaN,7.110855,5.78078,4.862534
min,NaN,-7.500000,-6.50000,-6.500000
25%,NaN,9.000000,7.00000,4.000000
50%,NaN,14.000000,11.00000,7.000000
75%,NaN,20.000000,16.00000,11.000000


In [ ]:
# Guarda una copia limpia en tu entorno de Colab
df.to_csv('dataset_limpio.csv', index=False)
# Y si quieres descargarlo a tu PC:
from google.colab import files
files.download('dataset_limpio.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>